In [0]:
dbutils.widgets.text("catalog", "QA_Assessment", "Catalog")
dbutils.widgets.text("schema", "Bronze", "Schema")

# Bronze Layer - Raw Data Ingestion

## Overview
This notebook implements the **Bronze layer** of the Medallion architecture for SmartRetail Corp.

## Strategy
* **Source**: CSV files in `/Volumes/QA_Assessment/Bronze/raw`
* **Datasets**: customers, products, orders, order_items
* **Approach**: Append-only storage with metadata tracking
* **Metadata Added**:
  * `ingestion_timestamp` - When the data was ingested
  * `source_file_name` - Source file path for lineage

## Tables Created
* `QA_Assessment.Bronze.bronze_customers`
* `QA_Assessment.Bronze.bronze_products`
* `QA_Assessment.Bronze.bronze_orders`
* `QA_Assessment.Bronze.bronze_order_items`

In [0]:
# Get parameters
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

# Configuration
volume_path = f"/Volumes/{catalog}/{schema}/raw"
target_catalog = catalog
target_schema = schema

print(f"Configuration:")
print(f"  Catalog: {catalog}")
print(f"  Schema: {schema}")
print(f"  Volume Path: {volume_path}")
print(f"  Target: {target_catalog}.{target_schema}")

In [0]:
from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime

def ingest_to_bronze(file_name, table_name, primary_keys=None):
    """
    Ingest CSV file to Bronze layer with metadata.
    
    Args:
        file_name: Name of CSV file (e.g., 'customers.csv')
        table_name: Target Bronze table name (e.g., 'bronze_customers')
        primary_keys: List of primary key columns for deduplication
    """
    source_path = f"{volume_path}/{file_name}"
    target_table = f"{target_catalog}.{target_schema}.{table_name}"
    
    print(f"\n{'='*60}")
    print(f"Ingesting: {file_name} -> {target_table}")
    print(f"{'='*60}")
    
    # Read CSV with schema inference
    df = spark.read.csv(source_path, header=True, inferSchema=True)
    
    # Add metadata columns
    df_with_metadata = df \
        .withColumn("ingestion_timestamp", current_timestamp()) \
        .withColumn("source_file_name", lit(source_path))
    
    # Show sample and schema
    print(f"\nSchema:")
    df_with_metadata.printSchema()
    print(f"\nSample Data (5 rows):")
    df_with_metadata.show(5, truncate=False)
    
    record_count = df_with_metadata.count()
    print(f"\nTotal Records: {record_count:,}")
    
    # Write to Bronze table (append mode for initial load)
    df_with_metadata.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(target_table)
    
    print(f"\n✅ Successfully ingested {record_count:,} records to {target_table}")
    
    return df_with_metadata

print("✅ Bronze ingestion function defined")

In [0]:
# Ingest customers.csv
df_customers = ingest_to_bronze(
    file_name="customers.csv",
    table_name="bronze_customers",
    primary_keys=["customer_id"]
)

In [0]:
# Ingest products.csv
df_products = ingest_to_bronze(
    file_name="products.csv",
    table_name="bronze_products",
    primary_keys=["product_id"]
)

In [0]:
# Ingest orders.csv
df_orders = ingest_to_bronze(
    file_name="orders.csv",
    table_name="bronze_orders",
    primary_keys=["order_id"]
)

In [0]:
# Ingest order_items.csv
df_order_items = ingest_to_bronze(
    file_name="order_items.csv",
    table_name="bronze_order_items",
    primary_keys=["order_item_id"]
)

In [0]:
# Display summary of Bronze tables
print("\n" + "="*60)
print("BRONZE LAYER SUMMARY")
print("="*60)

bronze_tables = [
    "bronze_customers",
    "bronze_products",
    "bronze_orders",
    "bronze_order_items"
]

for table in bronze_tables:
    full_table_name = f"{target_catalog}.{target_schema}.{table}"
    count = spark.table(full_table_name).count()
    print(f"{full_table_name}: {count:,} records")



In [0]:
# Optimize Bronze tables for better read performance
print("DELTA LAKE OPTIMIZATIONS")

bronze_tables = [
    "bronze_customers",
    "bronze_products",
    "bronze_orders",
    "bronze_order_items"
]

for table in bronze_tables:
    full_table = f"{target_catalog}.{target_schema}.{table}"
    
    # Run OPTIMIZE to compact small files
    spark.sql(f"OPTIMIZE {full_table}")
    
    # Run VACUUM to remove old versions of files
    spark.sql(f"VACUUM {full_table}")

